# Qwen2.5-1.5B decode speed: Colab T4 vs our CPU

**What this is:** the hand-written greedy decode loop from milestone 1, run on the
GPU Colab gives away free, to find out what the CPU numbers in this project cost
us.

**What this is not:** part of the deployed system. Nothing here is imported by the
app, and no result here changes a default. It exists to put a number on one
question — *how much of our latency is the absence of a GPU?*

### The loop being run

Ported from `llm-internals/transformer_walkthrough.py`, which lives beside the
repo rather than in it. Same model, same prompt, same greedy argmax, same
one-token-at-a-time feed with the KV cache carried forward. `model.generate()` is
deliberately unused: the point of the original was to see each decode step, and
the point here is to time the identical work on different hardware.

Two things are deliberately **different** from the original, and both are
necessary rather than cosmetic — see the cells below:

1. **`float16`, not `bfloat16`.** The original picks bf16 on CUDA. A T4 is Turing
   (compute capability 7.5) and has no bf16 tensor cores; you get slow emulation.
2. **`torch.cuda.synchronize()` around every timed step.** CUDA kernels are
   asynchronous. Timing them the way the CPU version does measures how fast
   Python can queue work, which on a GPU looks like thousands of tokens/sec and
   is a fiction.

### What it is compared against

| run | precision | runtime | tok/s |
|---|---|---|---|
| This laptop's CPU, same loop, measured 2026-09-12 | float32 | transformers | **2.67** |
| Same, without the KV cache (12 tokens) | float32 | transformers | 1.23 |
| M1's original record, 2026-09-04 (12 tokens) | float32 | transformers | 3.53 |
| `qwen2.5:1.5b` in the app's router step | Q4_K_M | Ollama / llama.cpp | ~13.9 |
| `qwen2.5:7b` writing an answer | Q4_K_M | Ollama / llama.cpp | 2.8 – 4.2 |

The last two rows are **not the same loop**: Ollama runs 4-bit quantized weights
through llama.cpp, which is a different runtime at a different precision. They are
here because they are the numbers the actual product runs at, not because they are
a controlled comparison. The controlled comparison is the first row — and the one
this notebook adds by running the same loop on Colab's own CPU, on the same
machine as the T4, which removes the hardware difference entirely.

---
## 1. Confirm there is actually a GPU

Free Colab hands out a GPU only if the runtime is set to one, and silently gives
you a CPU otherwise. A benchmark that quietly measures the wrong device is worse
than one that fails, so this cell stops rather than continues.

**If it fails:** Runtime → Change runtime type → Hardware accelerator → T4 GPU.

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or
      "nvidia-smi produced nothing -- probably no GPU attached")

import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No CUDA device.\n"
        "Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU, "
        "then run this cell again."
    )

major, minor = torch.cuda.get_device_capability()
props = torch.cuda.get_device_properties(0)

print(f"torch                {torch.__version__}")
print(f"CUDA                 {torch.version.cuda}")
print(f"device               {props.name}")
print(f"compute capability   {major}.{minor}")
print(f"VRAM                 {props.total_memory / 1e9:.1f} GB")
print(f"SMs                  {props.multi_processor_count}")

# bfloat16 needs Ampere (8.0+). On a T4 (7.5) torch will accept bf16 and then run
# it through emulation, which is slower than fp16 and would make this benchmark
# quietly wrong rather than loudly broken.
BF16_OK = major >= 8
GPU_DTYPE = torch.bfloat16 if BF16_OK else torch.float16
print(f"\nchosen GPU dtype     {GPU_DTYPE}"
      f"   ({'bf16 supported' if BF16_OK else 'no bf16 on this card -> fp16'})")

---
## 2. Dependencies

Colab ships torch and transformers already. This only reports what is there and
installs transformers if it is missing or ancient — pinning it would risk
dragging in a torch that does not match Colab's CUDA.

In [ ]:
import importlib
import subprocess     # also imported in cell 1; repeated so this cell stands alone

try:
    import transformers
    print(f"transformers {transformers.__version__} (preinstalled)")
except ImportError:
    print("installing transformers ...")
    subprocess.run(["pip", "install", "-q", "transformers>=4.44"], check=True)
    transformers = importlib.import_module("transformers")
    print(f"transformers {transformers.__version__}")

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
PROMPT = "In one sentence, what is a KV cache?"   # M1's default, kept verbatim
MAX_NEW_TOKENS = 24                              # M1's default, kept verbatim

---
## 3. The loop

A faithful port of `transformer_walkthrough.py`'s `load`, `tokenize`, prefill and
`greedy_decode`, with the two GPU-specific corrections described at the top.

`cache_layers` is copied as-is because it earns its keep: the KV cache object has
changed shape three times across transformers versions, and Colab's version is
whatever Colab decided today.

In [ ]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def cache_layers(past):
    """[(keys, values), ...] per layer, across three transformers cache APIs."""
    if past is None:
        return []
    if isinstance(past, tuple):                    # legacy tuple-of-tuples
        return list(past)
    layers = getattr(past, "layers", None)         # transformers >= ~4.54
    if layers is not None:
        return [(l.keys, l.values) for l in layers
                if getattr(l, "keys", None) is not None]
    if hasattr(past, "key_cache"):                 # ~4.36 - 4.53
        return list(zip(past.key_cache, past.value_cache))
    return []


def cache_stats(past):
    layers = cache_layers(past)
    if not layers:
        return 0, 0, None
    elems = sum(k.numel() + v.numel() for k, v in layers)
    nbytes = sum(k.numel() * k.element_size() + v.numel() * v.element_size()
                 for k, v in layers)
    return elems, nbytes, tuple(layers[0][0].shape)


def sync(device):
    """Wait for the GPU to actually finish.

    Without this, a timed CUDA section measures the time to *enqueue* work, not
    to do it -- every step looks like a fraction of a millisecond and the
    resulting tok/s is meaningless. The CPU path needs no equivalent, which is
    why the original loop has none.
    """
    if device == "cuda":
        torch.cuda.synchronize()


def load(device, dtype, model_id=MODEL_ID):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    try:
        model = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype)
    except TypeError:
        # `dtype=` is the current name; older transformers only knows torch_dtype.
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype)
    model = model.to(device)
    model.eval()
    return tokenizer, model


@torch.no_grad()
def run(device, dtype, label, max_new_tokens=MAX_NEW_TOKENS, verbose=True):
    """Prefill once, then decode one token at a time, timing every step."""
    torch.manual_seed(0)          # greedy is deterministic; pinned anyway
    print(f"\n{'=' * 74}\n{label}\n{'=' * 74}")

    t0 = time.perf_counter()
    tokenizer, model = load(device, dtype)
    sync(device)
    load_s = time.perf_counter() - t0
    print(f"load: {load_s:.1f}s   dtype={next(model.parameters()).dtype}   "
          f"params={sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

    # Same chat template as M1: an -Instruct model behaves like an assistant only
    # inside the markup it was tuned on.
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": PROMPT}],
        tokenize=False, add_generation_prompt=True)
    ids = tokenizer(text, return_tensors="pt").to(device)
    prompt_len = ids["input_ids"].shape[1]

    # ── warm-up, excluded from every number below ────────────────────────────
    # The first CUDA forward pass pays for context creation, kernel autotuning
    # and cuBLAS handles. Left in, it lands entirely on step 0 and inflates it by
    # an order of magnitude.
    _ = model(input_ids=ids["input_ids"][:, :1], use_cache=False)
    sync(device)

    # ── prefill: the whole prompt in one pass ────────────────────────────────
    sync(device)
    t0 = time.perf_counter()
    prefill = model(**ids, use_cache=True)
    sync(device)
    prefill_ms = (time.perf_counter() - t0) * 1000

    past = prefill.past_key_values
    next_logits = prefill.logits[0, -1, :]
    attention_mask = ids["attention_mask"]

    eos_ids = {tokenizer.eos_token_id,
               tokenizer.convert_tokens_to_ids("<|im_end|>")}
    eos_ids.discard(None)

    step_ms, generated = [], []
    if verbose:
        print(f"\n  {'step':>4}  {'token':<14} {'cache seq':>9}  "
              f"{'cache mem':>10}  {'ms':>7}")
        print("  " + "-" * 52)

    for step in range(max_new_tokens):
        next_id = int(torch.argmax(next_logits).item())
        generated.append(next_id)
        if next_id in eos_ids:
            if verbose:
                print(f"  {step:>4}  {tokenizer.decode([next_id])!r:<14} "
                      f"<- end of turn, stopping")
            generated.pop()
            break

        next_input = torch.tensor([[next_id]], device=device)
        attention_mask = torch.cat(
            [attention_mask,
             torch.ones((1, 1), dtype=attention_mask.dtype, device=device)],
            dim=-1)

        sync(device)
        t0 = time.perf_counter()
        out = model(input_ids=next_input, attention_mask=attention_mask,
                    past_key_values=past, use_cache=True)
        sync(device)
        dt = (time.perf_counter() - t0) * 1000

        step_ms.append(dt)
        past = out.past_key_values
        next_logits = out.logits[0, -1, :]

        _, nbytes, kshape = cache_stats(past)
        if verbose:
            print(f"  {step:>4}  {tokenizer.decode([next_id])!r:<14} "
                  f"{kshape[2] if kshape else 0:>9}  "
                  f"{nbytes / 1e6:>8.2f} MB  {dt:>7.1f}")

    total_ms = sum(step_ms)
    tok_s = len(step_ms) / (total_ms / 1000) if total_ms else float("nan")
    median = sorted(step_ms)[len(step_ms) // 2] if step_ms else float("nan")

    print(f"\n  prompt tokens      {prompt_len}")
    print(f"  generated tokens   {len(step_ms)}")
    print(f"  prefill            {prefill_ms:.1f} ms "
          f"({prompt_len / (prefill_ms / 1000):.0f} tok/s)")
    print(f"  decode total       {total_ms:.0f} ms")
    print(f"  per token          {total_ms / len(step_ms):.1f} ms "
          f"(median {median:.1f})")
    print(f"  DECODE THROUGHPUT  {tok_s:.2f} tok/s")
    print(f"\n  {tokenizer.decode(generated, skip_special_tokens=True)!r}")

    del model
    if device == "cuda":
        torch.cuda.empty_cache()

    return {"label": label, "device": device, "dtype": str(dtype),
            "tok_s": tok_s, "per_token_ms": total_ms / len(step_ms),
            "median_ms": median, "prefill_ms": prefill_ms,
            "prompt_tokens": prompt_len, "generated": len(step_ms),
            "text": tokenizer.decode(generated, skip_special_tokens=True)}

---
## 4. On the GPU

In [ ]:
# A dict, not a list: notebook cells get re-run out of order, and positional
# indexing into an accumulating list breaks silently when they do.
results = {}

# Run 1 -- M1's exact settings, 24 tokens, for the like-for-like comparison.
results["gpu_24"] = run("cuda", GPU_DTYPE, f"GPU — {GPU_DTYPE} — 24 tokens (M1 settings)")

# Run 2 -- the same loop, longer. At GPU speed 24 tokens is about one second of
# decode, which is too short a window to quote a throughput from: a single slow
# step moves it several percent. 128 tokens costs a few more seconds and is the
# number worth reporting.
results["gpu_128"] = run("cuda", GPU_DTYPE, f"GPU — {GPU_DTYPE} — 128 tokens",
                         max_new_tokens=128, verbose=False)

---
## 5. On Colab's own CPU

This is the comparison that actually isolates the hardware: the same loop, the
same model, the same prompt, on the same machine, differing only in device. The
laptop row in the table at the top differs in CPU *and* in everything else about
the box; this one does not.

`float32`, matching the original's CPU branch — bf16 matmuls on CPU are slow and
patchily supported, so fp32 is the honest CPU baseline rather than a handicap.

Slow: expect a couple of minutes.

In [ ]:
results["cpu_24"] = run("cpu", torch.float32, "Colab CPU — float32 — 24 tokens")

---
## 6. What the KV cache is worth, on a GPU

M1 measured cache vs no-cache on CPU (9,723 ms vs 4,519 ms for 12 tokens on this
laptop). The same comparison on a GPU answers a different question: whether the
cache still matters when the arithmetic is nearly free. Every step here re-runs
the whole sequence from scratch, so the work is O(n²) in total.

In [ ]:
@torch.no_grad()
def decode_without_cache(device, dtype, max_new_tokens=12):
    torch.manual_seed(0)
    tokenizer, model = load(device, dtype)

    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": PROMPT}],
        tokenize=False, add_generation_prompt=True)
    ids = tokenizer(text, return_tensors="pt").to(device)
    input_ids = ids["input_ids"].clone()

    _ = model(input_ids=input_ids[:, :1], use_cache=False)   # warm-up
    sync(device)

    print(f"  {'step':>4}  {'seq len':>8}  {'ms':>7}")
    print("  " + "-" * 24)
    total = 0.0
    for step in range(max_new_tokens):
        sync(device)
        t0 = time.perf_counter()
        out = model(input_ids=input_ids, use_cache=False)
        sync(device)
        dt = (time.perf_counter() - t0) * 1000
        total += dt
        next_id = int(torch.argmax(out.logits[0, -1, :]).item())
        input_ids = torch.cat(
            [input_ids, torch.tensor([[next_id]], device=device)], dim=-1)
        print(f"  {step:>4}  {input_ids.shape[1]:>8}  {dt:>7.1f}")

    print(f"\n  no cache: {total:.0f} ms for {max_new_tokens} tokens "
          f"({max_new_tokens / (total / 1000):.2f} tok/s)")
    del model
    if device == "cuda":
        torch.cuda.empty_cache()
    return total


print("=" * 74)
print("GPU, cache thrown away every step")
print("=" * 74)
nocache_total = decode_without_cache("cuda", GPU_DTYPE, 12)

# 12 x the mean per-token time from section 4, not a separate measurement of the
# first 12 steps. Per-step time is nearly flat at this context length so the two
# agree closely, but this is an estimate and should be read as one.
cached_12 = results["gpu_24"]["per_token_ms"] * 12
print(f"\n  cached, 12 x mean per-token from section 4: {cached_12:.0f} ms")
print(f"  uncached, measured:                        {nocache_total:.0f} ms")
print(f"  the cache is worth {nocache_total / cached_12:.1f}x here")

---
## 7. The comparison

In [ ]:
# Measured on the project's dev host with the same loop. See the table at the
# top of this notebook for why the Ollama rows are not a controlled comparison.
LAPTOP_CPU_TOK_S = 2.67          # 24 tokens in 8,995 ms, float32, 2026-09-12
OLLAMA_1_5B_TOK_S = 13.9         # Q4_K_M via llama.cpp, router step
OLLAMA_7B_TOK_S = 4.2            # Q4_K_M via llama.cpp, writing an answer

gpu = results["gpu_128"]      # the longer run: the number worth quoting
gpu_24 = results["gpu_24"]    # M1's exact settings: the like-for-like one
colab_cpu = results["cpu_24"]

rows = [
    ("Colab GPU",   gpu["dtype"].replace("torch.", ""),    "transformers, 128 tok", gpu["tok_s"]),
    ("Colab GPU",   gpu_24["dtype"].replace("torch.", ""), "transformers,  24 tok", gpu_24["tok_s"]),
    ("Colab CPU",   "float32", "transformers,  24 tok",    colab_cpu["tok_s"]),
    ("Laptop CPU",  "float32", "transformers,  24 tok",    LAPTOP_CPU_TOK_S),
    ("Laptop CPU",  "Q4_K_M",  "Ollama 1.5b",              OLLAMA_1_5B_TOK_S),
    ("Laptop CPU",  "Q4_K_M",  "Ollama 7b",                OLLAMA_7B_TOK_S),
]

print(f"{'where':<12} {'precision':<10} {'runtime':<23} {'tok/s':>8}  {'vs laptop fp32':>15}")
print("-" * 73)
for where, precision, runtime, tok_s in rows:
    print(f"{where:<12} {precision:<10} {runtime:<23} {tok_s:>8.2f}  "
          f"{tok_s / LAPTOP_CPU_TOK_S:>14.1f}x")

print(f"""
(the GPU figures below use the 128-token run)

Same loop, same machine, GPU vs CPU:  {gpu['tok_s'] / colab_cpu['tok_s']:.1f}x
Same loop, Colab GPU vs this laptop:  {gpu['tok_s'] / LAPTOP_CPU_TOK_S:.1f}x
GPU (fp16, transformers) vs what the
  product actually runs at (7b, Q4):  {gpu['tok_s'] / OLLAMA_7B_TOK_S:.1f}x
    -- but note that is a 1.5B model against a 7B one, so it is an upper bound
       on what a GPU would buy the product, not an estimate of it.
""")

# Greedy decoding is deterministic, so the same model on different hardware
# should produce the same tokens. If these differ, suspect the port rather than
# the hardware -- fp16 can flip an argmax that fp32 decides by a hair.
print("Same settings (24 tokens), both devices:")
print(f"  GPU: {gpu_24['text']!r}")
print(f"  CPU: {colab_cpu['text']!r}")
print(f"  identical: {gpu_24['text'] == colab_cpu['text']}")

---
## 8. Reading these numbers honestly

**What this measures.** One request, batch size 1, 24 tokens, a 39-token prompt,
in fp16/fp32 through transformers' Python-level loop. That is the same shape of
work the app's decode does, so the ratio is informative.

**What it does not measure, and must not be read as:**

- **Not the speedup the product would get.** This is 1.5B against the app's 7B,
  and unquantized transformers against llama.cpp's Q4_K_M kernels. Two variables
  move at once. Treat the GPU-vs-7B row as a ceiling, not a forecast.
- **Not throughput under load.** Batch size 1 leaves a GPU almost entirely idle;
  decode is memory-bandwidth bound and a single stream cannot saturate it. A
  server batching concurrent requests gets far more out of the same card, which
  is the entire argument for vLLM over a per-request loop.
- **Not a stable number.** Colab hands out whatever card is free, throttles, and
  shares the host CPU. Run it more than once before believing any single figure.
- **Nothing about VRAM headroom for the real system.** `qwen2.5:7b` at Q4_K_M is
  ~4.7GB of weights; this 1.5B model in fp16 is ~3.1GB. The deployment's VRAM
  question is answered in `docs/DEPLOYMENT.md`, not here.

**Why it was worth running anyway.** `docs/INFERENCE.md` established that on the
dev host generation is ~97% of request latency, at 2.8–4.2 tok/s. Every latency
decision in this project — the router split, the 512-token cap, raising `num_ctx`
for correctness while accepting +25% — follows from that one constraint. This
notebook puts a measured number on how much of that constraint is the hardware,
using the loop the project started from.